In [14]:
"""
Product-tariff crosswalk

Flags which of our 110 StatCan products were hit by Canada's March 2025 counter-tariffs,
so to later compare their price changes against products that weren't.

StatCan uses shelf names ("Chicken breasts"); the tariff list uses customs terms
("fowls of the species Gallus domesticus"), so the two can't be joined directly.

Method:
- Each product was assigned the HS code (first 6 digits) it falls under.
- Every code was checked against the tariff list line by line, because some
  categories are only partly tariffed (e.g. 0210 covers smoked poultry, not bacon).
- Products not matched are treated as untariffed.
"""
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)

prices = pd.read_csv("../data/raw/retail_prices/18100245.csv", encoding="utf-8-sig", low_memory=False)
our_products = sorted(prices["Products"].str.strip().unique())

t2025 = pd.read_csv("../data/raw/tariffs/counter_tariffs_2025_03.csv", dtype=str)

print(len(our_products), "products")
print(len(t2025), "tariff lines")

110 products
1256 tariff lines


In [ ]:
# StatCan's 110 products, A to Z
for product in our_products:
    print(product)

Almonds, 200 grams
Apple juice, 2 litres
Apples, per kilogram
Avocado, unit
Baby food, 128 millilitres
Bacon, 500 grams
Bananas, per kilogram
Beef rib cuts, per kilogram
Beef stewing cuts, per kilogram
Beef striploin cuts, per kilogram
Beef top sirloin cuts, per kilogram
Block cheese, 500 grams
Broccoli, unit
Brown rice, 900 grams
Butter, 454 grams
Cabbage, per kilogram
Canned baked beans, 398 millilitres
Canned beans and lentils, 540 millilitres
Canned corn, 341 millilitres
Canned peach, 398 millilitres
Canned pear, 398 millilitres
Canned salmon, 213 grams
Canned soup, 284 millilitres
Canned tomatoes, 796 millilitres
Canned tuna, 170 grams
Canola oil, 3 litres
Cantaloupe, unit
Carrots, 1.36 kilograms
Celery, unit
Cereal, 400 grams
Chicken breasts, per kilogram
Chicken drumsticks, per kilogram
Chicken thigh, per kilogram
Cookies and sweet biscuits, 300 grams
Crackers and crisp breads, 200 grams
Cream, 1 litre
Cucumber, unit
Deodorant, 85 grams
Dried lentils, 900 grams
Dry beans and leg

In [3]:
# Food (02-24) and personal-item (33-34) categories on the 2025 tariff list
t2025["hs_category_code"] = t2025["tariff_item"].str[:4]

food_and_personal_items_codes = [f"{n:02d}" for n in range(2, 25)] + ["33", "34"]
food_and_personal_items = t2025[t2025["hs_category_code"].str[:2].isin(food_and_personal_items_codes)]

tariff_categories_2025 = (
    food_and_personal_items.groupby("hs_category_code")["hs_heading"]
    .first().reset_index()
    .rename(columns={"hs_heading": "category_description"})
)
print(len(tariff_categories_2025), "categories")
tariff_categories_2025

78 categories


,hs_category_code,category_description
0,0207,"Meat and edible offal, of the poultry of heading 01.05, fresh, chilled or frozen."
1,0209,"Pig fat, free of lean meat, and poultry fat, not rendered or otherwise extracted, fresh, chilled, frozen, salted, in..."
2,0210,"Meat and edible meat offal, salted, in brine, dried or smoked; edible flours and meals of meat or meat offal."
3,0401,"Milk and cream, not concentrated nor containing added sugar or other sweetening matter."
4,0402,"Milk and cream, concentrated or containing added sugar or other sweetening matter."
5,0403,"Yogurt; buttermilk, curdled milk and cream, kephir and other fermented or acidified milk and cream, whether or not c..."
6,0404,"Whey, whether or not concentrated or containing added sugar or other sweetening matter; products consisting of natur..."
7,0405,Butter and other fats and oils derived from milk; dairy spreads.
8,0406,Cheese and curd.
9,0407,"Birds' eggs, in shell, fresh, preserved or cooked."


In [ ]:
# Function to display every tariff line in one category
def show_category(category_code):
    lines = food_and_personal_items[food_and_personal_items["hs_category_code"] == category_code]
    return lines[["tariff_item", "description"]]

In [5]:
# Check: 0210 (salted/smoked meat) only lists poultry, so bacon (pork) is not tariffed
show_category("0210")

,tariff_item,description
34,0210.99.11,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of fowls of the species Gal..."
35,0210.99.12,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of fowls of the species Gal..."
36,0210.99.13,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of fowls of the species Gal..."
37,0210.99.14,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of turkeys, within access c..."
38,0210.99.15,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of turkeys, over access com..."
39,0210.99.16,"Other, including edible flours and meals of meat or meat offal: Other - Meat of poultry: Of turkeys, over access com..."


In [6]:
# Check: 0811 (frozen fruit) only lists raspberries/blackberries, so frozen strawberries are not tariffed
show_category("0811")

,tariff_item,description
109,0811.20.00,"Raspberries, blackberries, mulberries, loganberries, black, white or red currants and gooseberries"


In [7]:
# Check: 2103 (sauces) lists ketchup, tomato sauces, mayo and dressing, but not salsa
show_category("2103")

,tariff_item,description
271,2103.10.00,Soya sauce
272,2103.20.10,Tomato ketchup and other tomato sauces - Tomato ketchup
273,2103.20.90,Tomato ketchup and other tomato sauces - Other
274,2103.30.10,Mustard flour and meal and prepared mustard - Mustard flour and meal
275,2103.30.20,Mustard flour and meal and prepared mustard - Prepared mustard
276,2103.90.10,Other - Mayonnaise and salad dressing


In [ ]:
# Products hit by the March 2025 tariffs, with their HS code (first 6 digits)
tariffed_products = {
    # Dairy and eggs
    "Milk, 1 litre": "0401.20",
    "Milk, 2 litres": "0401.20",
    "Milk, 4 litres": "0401.20",
    "Cream, 1 litre": "0401.50",
    "Yogurt, 500 grams": "0403.20",
    "Butter, 454 grams": "0405.10",
    "Block cheese, 500 grams": "0406.90",
    "Eggs, 1 dozen": "0407.21",
    # Meat
    "Whole chicken, per kilogram": "0207.11",
    "Chicken breasts, per kilogram": "0207.13",
    "Chicken drumsticks, per kilogram": "0207.13",
    "Chicken thigh, per kilogram": "0207.13",
    "Wieners, 400 grams": "1601.00",
    # Fruit and vegetables
    "Tomatoes, per kilogram": "0702.00",
    "Oranges, per kilogram": "0805.10",
    "Oranges, 1.36 kilograms": "0805.10",
    "Lemons, unit": "0805.50",
    "Limes, unit": "0805.50",
    "Orange juice, 2 litres": "2009.12",
    # Coffee and tea
    "Roasted or ground coffee, 340 grams": "0901.21",
    "Tea (20 bags)": "0902.30",
    # Staples
    "Brown rice, 900 grams": "1006.20",
    "White rice, 2 kilograms": "1006.30",
    "Wheat flour, 2.5 kilograms": "1101.00",
    "White sugar, 2 kilograms": "1701.99",
    "Dry or fresh pasta, 500 grams": "1902.19",
    "Canola oil, 3 litres": "1514.19",
    "Margarine, 907 grams": "1517.10",
    # Prepared foods and sauces
    "Peanut butter, 1 kilogram": "2008.11",
    "Peanuts, 450 grams": "2008.11",
    "Ketchup, 1 litre": "2103.20",
    "Pasta sauce, 650 millilitres": "2103.20",
    "Mayonnaise, 890 millilitres": "2103.90",
    "Salad dressing, 475 millilitres": "2103.90",
    "Canned soup, 284 millilitres": "2104.10",
    "Frozen pizza, 390 grams": "1905.90",
    # Personal items
    "Shampoo, 400 millilitres": "3305.10",
    "Toothpaste, 100 millilitres": "3306.10",
    "Deodorant, 85 grams": "3307.20",
}
print(len(tariffed_products), "tariffed products")

39 tariffed products


In [ ]:
# Check if every product name typed exists in list
typos = set(tariffed_products) - set(our_products)
print("Names not found:", typos if typos else "none")

Names not found: none


In [17]:
# Check if every product name we typed exists in list
typos = set(tariffed_products) - set(our_products)
print("Names not found:", typos if typos else "none")

Names not found: none


In [18]:
# Build final crosswalk with one row per product with its status and HS code
crosswalk = pd.DataFrame({"product": our_products})

crosswalk["status"] = "not tariffed"
crosswalk.loc[crosswalk["product"].isin(tariffed_products), "status"] = "tariffed"
crosswalk["hs_code"] = crosswalk["product"].map(tariffed_products).fillna("")

print(crosswalk["status"].value_counts())
crosswalk

status
not tariffed    71
tariffed        39
Name: count, dtype: int64


,product,status,hs_code
0,"Almonds, 200 grams",not tariffed,
1,"Apple juice, 2 litres",not tariffed,
2,"Apples, per kilogram",not tariffed,
3,"Avocado, unit",not tariffed,
4,"Baby food, 128 millilitres",not tariffed,
5,"Bacon, 500 grams",not tariffed,
6,"Bananas, per kilogram",not tariffed,
7,"Beef rib cuts, per kilogram",not tariffed,
8,"Beef stewing cuts, per kilogram",not tariffed,
9,"Beef striploin cuts, per kilogram",not tariffed,


In [19]:
# Save crosswalk to CSV
crosswalk.to_csv("../data/reference/product_tariff_crosswalk.csv", index=False)
print("Saved", len(crosswalk), "rows")

Saved 110 rows
